<a href="https://colab.research.google.com/github/vitor-thompson/Python-para-financas-investimento-e-analise-de-dados./blob/main/redimento_do_portifolio_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# --- Configurações Iniciais ---
data_inicial = pd.to_datetime("2025-12-15")
quantidade_cotas_ini = 44.581

# DEFINE O DIA ATUAL AUTOMATICAMENTE
data_final = pd.to_datetime(datetime.now().date())

# --- MODELO PARA NOVAS ADIÇÕES (Copie e cole abaixo) ---
# ("AAAA-MM-DD", "TICKER.SA", QUANTIDADE, PRECO_UNITARIO),
# Use QUANTIDADE positiva para COMPRAS e negativa para VENDAS.

movimentacoes = [
    # Bloco 1: Carteira inicial
    ("2025-12-15", "ITUB4.SA", 1, 40.14), ("2025-12-15", "ITSA4.SA", 2, 11.94),
    ("2025-12-15", "TAEE11.SA", 2, 42.99), ("2025-12-15", "PETR4.SA", 3, 31.91),
    ("2025-12-15", "RADL3.SA", 4, 25.34), ("2025-12-15", "WEGE3.SA", 2, 49.36),

    # Bloco 2: Janeiro
    ("2026-01-15", "ITSA4.SA", 1, 12.09), ("2026-01-15", "TAEE11.SA", 1, 39.99),
    ("2026-01-15", "PETR4.SA", 1, 31.66), ("2026-01-15", "RADL3.SA", 2, 24.23),
    ("2026-01-15", "WEGE3.SA", 1, 46.85),

    # Bloco 3: Fevereiro
    ("2026-02-19", "TAEE11.SA", 1, 44.27), ("2026-02-19", "PETR4.SA", 1, 37.74),
    ("2026-02-19", "RADL3.SA", 1, 26.73), ("2026-02-19", "WEGE3.SA", 1, 51.62),

    # Bloco 4: Março
    ("2026-03-16", "ITUB4.SA", 5, 43.07), ("2026-03-16", "ITSA4.SA", 5, 13.42),
    ("2026-03-16", "BBSE3.SA", 3, 34.90), ("2026-03-16", "TAEE11.SA", -2, 42.77),
    ("2026-03-16", "PETR4.SA", -2, 45.50), ("2026-03-16", "WEGE3.SA", -2, 45.62),
    ("2026-03-17", "ITUB4.SA", -5, 42.97),

    # --- ADICIONE NOVAS MOVIMENTAÇÕES ABAIXO DESTA LINHA ---
    # Exemplo: ("2026-03-20", "VALE3.SA", 10, 65.00),
]

def calcular_carteira_dinamica(movs, data_alvo):
    df_movs = pd.DataFrame(movs, columns=['data', 'ticker', 'quant', 'preco'])
    df_movs['data'] = pd.to_datetime(df_movs['data'])

    tickers = df_movs['ticker'].unique()

    # Busca precos do início até hoje (adicionamos 1 dia ao end por segurança do yfinance)
    data_fim_busca = (data_alvo + timedelta(days=1)).strftime('%Y-%m-%d')
    precos = yf.download(list(tickers), start="2025-12-15", end=data_fim_busca)['Close']

    # Se baixar apenas um ticker, o pandas retorna Series, forçamos DataFrame
    if isinstance(precos, pd.Series):
        precos = precos.to_frame(name=tickers[0])

    cotas_atuais = quantidade_cotas_ini
    datas_eventos = sorted(df_movs[df_movs['data'] > "2025-12-15"]['data'].unique())

    # 1. Processamento de Cotização por Evento
    for data_evento in datas_eventos:
        posicao_antes = df_movs[df_movs['data'] < data_evento].groupby('ticker')['quant'].sum()

        # Encontra o último dia com pregão antes do evento
        data_previa_idx = precos.index[precos.index < data_evento]
        if len(data_previa_idx) == 0: continue

        data_previa = data_previa_idx[-1]
        precos_vespera = precos.loc[data_previa]

        patrimonio_antes = sum(posicao_antes[t] * precos_vespera[t] for t in posicao_antes.index if t in precos_vespera and not pd.isna(precos_vespera[t]))
        valor_cota_viva = patrimonio_antes / cotas_atuais

        df_hoje = df_movs[df_movs['data'] == data_evento]
        fluxo_caixa = (df_hoje['quant'] * df_hoje['preco']).sum()

        # O número de cotas só muda se houver aporte/resgate financeiro
        novas_cotas = fluxo_caixa / valor_cota_viva
        cotas_atuais += novas_cotas

    # 2. Cálculo Final (Baseado no último fechamento disponível)
    data_final_efetiva = precos.index[-1]
    precos_finais = precos.iloc[-1]

    posicao_final = df_movs.groupby('ticker')['quant'].sum()
    patrimonio_final = sum(posicao_final[t] * precos_finais[t] for t in posicao_final.index if t in precos_finais and not pd.isna(precos_finais[t]))

    valor_cota_final = patrimonio_final / cotas_atuais
    total_investido_liquido = (df_movs['quant'] * df_movs['preco']).sum()

    return {
        "Data de Fechamento": data_final_efetiva.date(),
        "Patrimônio Total R$": patrimonio_final,
        "Total de Cotas": cotas_atuais,
        "Valor da Cota Final": valor_cota_final,
        "Lucro/Prejuízo Total R$": patrimonio_final - total_investido_liquido
    }

# Execução
resultado = calcular_carteira_dinamica(movimentacoes, data_final)

print("-" * 45)
print(f"RELATÓRIO DE CARTEIRA ATUALIZADO - {resultado['Data de Fechamento']}")
print("-" * 45)
for k, v in resultado.items():
    if k != "Data de Fechamento":
        print(f"{k:<25}: {v:.4f}" if "Cota" in k else f"{k:<25}: R$ {v:,.2f}")
print("-" * 45)

/tmp/ipykernel_230/186404792.py:49: FutureWarning: YF.download() has changed argument auto_adjust default to True
  precos = yf.download(list(tickers), start="2025-12-15", end=data_fim_busca)['Close']
[*********************100%***********************]  7 of 7 completed

---------------------------------------------
RELATÓRIO DE CARTEIRA ATUALIZADO - 2026-03-24
---------------------------------------------
Patrimônio Total R$      : R$ 733.97
Total de Cotas           : 68.9377
Valor da Cota Final      : 10.6469
Lucro/Prejuízo Total R$  : R$ 44.23
---------------------------------------------
